# RiskFlow PayGuard — IEEE-CIS Fraud-Risk EDA

## Phase 1 objective

This notebook examines the IEEE-CIS Fraud Detection dataset from the perspective
of a production fraud-risk product.

The analysis focuses on:

- source-table contracts
- target imbalance
- identity-data coverage
- missingness and cardinality
- transaction amount behavior
- chronological fraud patterns
- fraud-rate segmentation
- baseline feature selection

Raw Kaggle files remain local and are not committed to Git.


## 1. Setup

The transaction table is the primary table. Identity data is optional and is
linked through `TransactionID`.

The complete transaction and identity tables are loaded separately in this
section. A full wide join is intentionally avoided until it is analytically
necessary.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

from src.data_processing import (
    AMOUNT_COLUMN,
    JOIN_KEY,
    TARGET_COLUMN,
    TIME_COLUMN,
    load_train_tables,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

print(f"Project root: {PROJECT_ROOT.name}")
print(f"Raw data:     {DATA_DIR.relative_to(PROJECT_ROOT)}")
print(f"Processed:    {PROCESSED_DIR.relative_to(PROJECT_ROOT)}")

Project root: riskflow-payguard
Raw data:     data/raw
Processed:    data/processed


## 2. Load the training source tables

Set `NROWS` to an integer such as `100_000` for quick development.

Use `None` for the complete Phase 1 analysis.


In [2]:
NROWS: int | None = None

transaction, identity = load_train_tables(
    DATA_DIR,
    nrows=NROWS,
)

print(f"Transaction shape: {transaction.shape}")
print(f"Identity shape:    {identity.shape}")

Transaction shape: (590540, 394)
Identity shape:    (144233, 41)


## 3. Dataset dimensions and join coverage

Identity attributes exist for only a subset of transactions. Their absence may
be operationally meaningful, so identity availability should be measured before
deciding how missing identity values will be handled.


In [3]:
transaction_memory_mb = (
    transaction.memory_usage(deep=True).sum() / 1024**2
)
identity_memory_mb = (
    identity.memory_usage(deep=True).sum() / 1024**2
)

identity_key_set = set(identity[JOIN_KEY])
has_identity = transaction[JOIN_KEY].isin(identity_key_set)

identity_coverage = has_identity.mean()
orphan_identity_count = int(
    (~identity[JOIN_KEY].isin(transaction[JOIN_KEY])).sum()
)

dataset_dimensions = pd.DataFrame(
    [
        {
            "table": "train_transaction",
            "rows": len(transaction),
            "columns": transaction.shape[1],
            "unique_transaction_ids": transaction[JOIN_KEY].nunique(),
            "memory_mb": transaction_memory_mb,
        },
        {
            "table": "train_identity",
            "rows": len(identity),
            "columns": identity.shape[1],
            "unique_transaction_ids": identity[JOIN_KEY].nunique(),
            "memory_mb": identity_memory_mb,
        },
    ]
)

display(dataset_dimensions)

print(f"Identity coverage:       {identity_coverage:.2%}")
print(f"Transactions with ID:    {has_identity.sum():,}")
print(f"Transactions without ID: {(~has_identity).sum():,}")
print(f"Orphan identity rows:    {orphan_identity_count:,}")

,table,rows,columns,unique_transaction_ids,memory_mb
0,train_transaction,590540,394,590540,"1,791.6674"
1,train_identity,144233,41,144233,56.5075


Identity coverage:       24.42%
Transactions with ID:    144,233
Transactions without ID: 446,307
Orphan identity rows:    0


In [4]:
assert transaction[JOIN_KEY].is_unique
assert identity[JOIN_KEY].is_unique
assert orphan_identity_count == 0
assert TARGET_COLUMN in transaction.columns
assert transaction[TARGET_COLUMN].notna().all()
assert set(transaction[TARGET_COLUMN].unique()).issubset({0, 1})

print("Source-table contract checks passed.")

Source-table contract checks passed.


## 4. Target imbalance

Fraud detection is an imbalanced classification problem. Accuracy would provide
a misleading view of model quality because legitimate transactions dominate the
dataset.

The initial model evaluation will therefore emphasize ranking, probability
quality, and fraud-capture metrics rather than accuracy.


In [5]:
target_counts = (
    transaction[TARGET_COLUMN]
    .value_counts()
    .reindex([0, 1], fill_value=0)
)

legitimate_count = int(target_counts.loc[0])
fraud_count = int(target_counts.loc[1])
total_count = int(target_counts.sum())

fraud_rate = fraud_count / total_count
legitimate_to_fraud_ratio = (
    legitimate_count / fraud_count if fraud_count else float("inf")
)

target_summary = pd.DataFrame(
    [
        {
            "class": "legitimate",
            "target_value": 0,
            "transaction_count": legitimate_count,
            "share": legitimate_count / total_count,
        },
        {
            "class": "fraud",
            "target_value": 1,
            "transaction_count": fraud_count,
            "share": fraud_rate,
        },
    ]
)

display(target_summary)

print(f"Total transactions:          {total_count:,}")
print(f"Fraudulent transactions:     {fraud_count:,}")
print(f"Overall fraud rate:          {fraud_rate:.4%}")
print(
    "Legitimate-to-fraud ratio: "
    f"{legitimate_to_fraud_ratio:,.1f}:1"
)

,class,target_value,transaction_count,share
0,legitimate,0,569877,0.9650
1,fraud,1,20663,0.0350


Total transactions:          590,540
Fraudulent transactions:     20,663
Overall fraud rate:          3.4990%
Legitimate-to-fraud ratio: 27.6:1


### Initial product implication

The observed imbalance will inform:

- `scale_pos_weight` or equivalent class weighting
- PR AUC and ROC AUC reporting
- precision and recall at decision thresholds
- manual-review capacity analysis
- fraud-capture and false-positive trade-offs

Resampling methods such as SMOTE are deferred until the chronological baseline
has been evaluated.


## 5. Missingness, data types, and cardinality

The IEEE-CIS dataset contains hundreds of sparse, anonymous, categorical, and
identifier-like fields.

This section creates a reusable profile for each source column without filling
missing values or joining the full transaction and identity tables.

The profile records:

- data type
- missing count and percentage
- non-null count
- distinct-value count
- distinct-value ratio
- cardinality band
- initial modeling-role hint


In [6]:
def classify_missingness(missing_pct: float) -> str:
    if missing_pct == 0:
        return "complete"
    if missing_pct <= 10:
        return "low_0_10"
    if missing_pct <= 50:
        return "moderate_10_50"
    if missing_pct <= 90:
        return "high_50_90"
    return "extreme_over_90"


def classify_cardinality(
    unique_count: int,
    unique_ratio: float,
) -> str:
    if unique_count <= 1:
        return "constant"
    if unique_ratio >= 0.98 and unique_count >= 1_000:
        return "identifier_like"
    if unique_count <= 20:
        return "low"
    if unique_count <= 100:
        return "medium"
    if unique_count <= 10_000:
        return "high"
    return "very_high"


def infer_role(
    column: str,
    series: pd.Series,
    unique_count: int,
) -> str:
    if column == JOIN_KEY:
        return "primary_key"
    if column == TARGET_COLUMN:
        return "target"
    if column == TIME_COLUMN:
        return "relative_time"
    if column == AMOUNT_COLUMN:
        return "monetary"

    if column.startswith("card"):
        return "categorical_code"

    if column in {"addr1", "addr2"}:
        return "categorical_code"

    if (
        pd.api.types.is_object_dtype(series.dtype)
        or pd.api.types.is_string_dtype(series.dtype)
        or pd.api.types.is_bool_dtype(series.dtype)
    ):
        return "categorical"

    if unique_count <= 100:
        return "discrete_numeric"

    return "continuous_numeric"

def build_column_profile(
    dataframe: pd.DataFrame,
    table_name: str,
) -> pd.DataFrame:
    row_count = len(dataframe)
    records: list[dict[str, object]] = []

    for column in dataframe.columns:
        series = dataframe[column]

        non_null_count = int(series.notna().sum())
        missing_count = row_count - non_null_count
        missing_pct = (
            100 * missing_count / row_count if row_count else 0.0
        )

        unique_count = int(series.nunique(dropna=True))
        unique_ratio = (
            unique_count / non_null_count if non_null_count else 0.0
        )

        records.append(
            {
                "table": table_name,
                "column": column,
                "dtype": str(series.dtype),
                "row_count": row_count,
                "non_null_count": non_null_count,
                "missing_count": missing_count,
                "missing_pct": missing_pct,
                "unique_count": unique_count,
                "unique_ratio": unique_ratio,
                "missingness_band": classify_missingness(
                    missing_pct
                ),
                "cardinality_band": classify_cardinality(
                    unique_count,
                    unique_ratio,
                ),
                "role_hint": infer_role(
                    column,
                    series,
                    unique_count,
                ),
            }
        )

    return pd.DataFrame.from_records(records)

In [7]:
transaction_profile = build_column_profile(
    transaction,
    "train_transaction",
)
identity_profile = build_column_profile(
    identity,
    "train_identity",
)

column_profile = pd.concat(
    [transaction_profile, identity_profile],
    ignore_index=True,
)

print(f"Profiled columns: {len(column_profile):,}")
print(
    "Transaction columns: "
    f"{len(transaction_profile):,}"
)
print(
    "Identity columns:    "
    f"{len(identity_profile):,}"
)

display(
    column_profile
    .sort_values(
        ["missing_pct", "unique_count"],
        ascending=[False, False],
    )
    .head(25)
)

Profiled columns: 435
Transaction columns: 394
Identity columns:    41


,table,column,dtype,row_count,non_null_count,missing_count,missing_pct,unique_count,unique_ratio,missingness_band,cardinality_band,role_hint
418,train_identity,id_24,float64,144233,4747,139486,96.7088,12,0.0025,extreme_over_90,low,discrete_numeric
419,train_identity,id_25,float64,144233,5132,139101,96.4419,341,0.0664,extreme_over_90,high,continuous_numeric
402,train_identity,id_08,float64,144233,5155,139078,96.4259,94,0.0182,extreme_over_90,medium,discrete_numeric
401,train_identity,id_07,float64,144233,5155,139078,96.4259,84,0.0163,extreme_over_90,medium,discrete_numeric
415,train_identity,id_21,float64,144233,5159,139074,96.4231,490,0.0950,extreme_over_90,high,continuous_numeric
420,train_identity,id_26,float64,144233,5163,139070,96.4204,95,0.0184,extreme_over_90,medium,discrete_numeric
416,train_identity,id_22,float64,144233,5169,139064,96.4162,25,0.0048,extreme_over_90,medium,discrete_numeric
417,train_identity,id_23,str,144233,5169,139064,96.4162,3,0.0006,extreme_over_90,low,categorical
421,train_identity,id_27,str,144233,5169,139064,96.4162,2,0.0004,extreme_over_90,low,categorical
14,train_transaction,dist2,float64,590540,37627,552913,93.6284,1751,0.0465,extreme_over_90,high,continuous_numeric


In [8]:
missingness_order = [
    "complete",
    "low_0_10",
    "moderate_10_50",
    "high_50_90",
    "extreme_over_90",
]

missingness_summary = (
    column_profile
    .assign(
        missingness_band=pd.Categorical(
            column_profile["missingness_band"],
            categories=missingness_order,
            ordered=True,
        )
    )
    .groupby(
        ["table", "missingness_band"],
        observed=False,
    )
    .size()
    .rename("column_count")
    .reset_index()
)

display(missingness_summary)

extremely_sparse = column_profile.loc[
    column_profile["missing_pct"] > 90,
    [
        "table",
        "column",
        "dtype",
        "missing_pct",
        "unique_count",
        "role_hint",
    ],
].sort_values(
    ["table", "missing_pct"],
    ascending=[True, False],
)

print(
    "Columns with more than 90% missingness: "
    f"{len(extremely_sparse)}"
)
display(extremely_sparse.head(30))

,table,missingness_band,column_count
0,train_identity,complete,3
1,train_identity,low_0_10,16
2,train_identity,moderate_10_50,10
3,train_identity,high_50_90,3
4,train_identity,extreme_over_90,9
5,train_transaction,complete,20
6,train_transaction,low_0_10,92
7,train_transaction,moderate_10_50,108
8,train_transaction,high_50_90,172
9,train_transaction,extreme_over_90,2


Columns with more than 90% missingness: 11


,table,column,dtype,missing_pct,unique_count,role_hint
418,train_identity,id_24,float64,96.7088,12,discrete_numeric
419,train_identity,id_25,float64,96.4419,341,continuous_numeric
401,train_identity,id_07,float64,96.4259,84,discrete_numeric
402,train_identity,id_08,float64,96.4259,94,discrete_numeric
415,train_identity,id_21,float64,96.4231,490,continuous_numeric
420,train_identity,id_26,float64,96.4204,95,discrete_numeric
416,train_identity,id_22,float64,96.4162,25,discrete_numeric
417,train_identity,id_23,str,96.4162,3,categorical
421,train_identity,id_27,str,96.4162,2,categorical
14,train_transaction,dist2,float64,93.6284,1751,continuous_numeric


In [9]:
cardinality_order = [
    "constant",
    "low",
    "medium",
    "high",
    "very_high",
    "identifier_like",
]

cardinality_summary = (
    column_profile
    .assign(
        cardinality_band=pd.Categorical(
            column_profile["cardinality_band"],
            categories=cardinality_order,
            ordered=True,
        )
    )
    .groupby(
        ["table", "cardinality_band"],
        observed=False,
    )
    .size()
    .rename("column_count")
    .reset_index()
)

display(cardinality_summary)

constant_columns = column_profile.loc[
    column_profile["cardinality_band"] == "constant",
    [
        "table",
        "column",
        "dtype",
        "missing_pct",
        "unique_count",
    ],
]

identifier_like_columns = column_profile.loc[
    column_profile["cardinality_band"] == "identifier_like",
    [
        "table",
        "column",
        "dtype",
        "missing_pct",
        "unique_count",
        "unique_ratio",
        "role_hint",
    ],
]

print(f"Constant columns:        {len(constant_columns)}")
print(
    "Identifier-like columns: "
    f"{len(identifier_like_columns)}"
)

display(constant_columns)
display(identifier_like_columns)

,table,cardinality_band,column_count
0,train_identity,constant,0
1,train_identity,low,17
2,train_identity,medium,12
3,train_identity,high,10
4,train_identity,very_high,1
5,train_identity,identifier_like,1
6,train_transaction,constant,0
7,train_transaction,low,151
8,train_transaction,medium,99
9,train_transaction,high,122


Constant columns:        0
Identifier-like columns: 2


,table,column,dtype,missing_pct,unique_count


,table,column,dtype,missing_pct,unique_count,unique_ratio,role_hint
0,train_transaction,TransactionID,int64,0.0000,590540,1.0000,primary_key
394,train_identity,TransactionID,int64,0.0000,144233,1.0000,primary_key


In [10]:
selected_risk_fields = [
    "ProductCD",
    "card1",
    "card4",
    "card6",
    "addr1",
    "addr2",
    "P_emaildomain",
    "R_emaildomain",
    "DeviceType",
    "DeviceInfo",
]

selected_field_profile = (
    column_profile.loc[
        column_profile["column"].isin(selected_risk_fields),
        [
            "table",
            "column",
            "dtype",
            "missing_pct",
            "unique_count",
            "unique_ratio",
            "cardinality_band",
            "role_hint",
        ],
    ]
    .sort_values(["table", "column"])
    .reset_index(drop=True)
)

display(selected_field_profile)

,table,column,dtype,missing_pct,unique_count,unique_ratio,cardinality_band,role_hint
0,train_identity,DeviceInfo,str,17.7262,1786,0.0151,high,categorical
1,train_identity,DeviceType,str,2.3732,2,0.0000,low,categorical
2,train_transaction,P_emaildomain,str,15.9949,59,0.0001,medium,categorical
3,train_transaction,ProductCD,str,0.0000,5,0.0000,low,categorical
4,train_transaction,R_emaildomain,str,76.7516,60,0.0004,medium,categorical
5,train_transaction,addr1,float64,11.1264,332,0.0006,high,categorical_code
6,train_transaction,addr2,float64,11.1264,74,0.0001,medium,categorical_code
7,train_transaction,card1,int64,0.0000,13553,0.0230,very_high,categorical_code
8,train_transaction,card4,str,0.2670,4,0.0000,low,categorical_code
9,train_transaction,card6,str,0.2660,4,0.0000,low,categorical_code


### Initial modeling implications

The profile supports several early rules:

- columns with extreme missingness require explicit justification
- constant columns provide no model signal
- identifier-like columns should not be used blindly
- high-cardinality categoricals require controlled encoding
- missingness may itself contain fraud-risk information
- identity absence should remain available as an explicit feature

No imputation or feature removal is performed in this section.


## 6. Transaction amount behavior

`TransactionAmt` is the main monetary exposure field.

This section examines:

- data-quality constraints
- distribution and skew
- percentile-based extremes
- legitimate versus fraudulent transaction values
- fraud rate by amount band
- share of total transaction value associated with fraud

Amount bands are descriptive EDA tools, not final production thresholds.


In [11]:
import numpy as np


amount = transaction[AMOUNT_COLUMN]

missing_amount_count = int(amount.isna().sum())
negative_amount_count = int((amount < 0).sum())
zero_amount_count = int((amount == 0).sum())
infinite_amount_count = int(
    np.isinf(amount.dropna().to_numpy()).sum()
)

print(f"Missing amounts:  {missing_amount_count:,}")
print(f"Negative amounts: {negative_amount_count:,}")
print(f"Zero amounts:     {zero_amount_count:,}")
print(f"Infinite amounts: {infinite_amount_count:,}")

assert missing_amount_count == 0
assert negative_amount_count == 0
assert infinite_amount_count == 0

print("Transaction amount quality checks passed.")

Missing amounts:  0
Negative amounts: 0
Zero amounts:     0
Infinite amounts: 0
Transaction amount quality checks passed.


In [12]:
amount_percentiles = [
    0.00,
    0.01,
    0.05,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.99,
    0.999,
    1.00,
]

amount_distribution = (
    amount
    .quantile(amount_percentiles)
    .rename_axis("quantile")
    .rename("transaction_amount")
    .reset_index()
)

amount_distribution["percentile"] = (
    amount_distribution["quantile"] * 100
)

amount_distribution = amount_distribution[
    ["percentile", "transaction_amount"]
]

display(amount_distribution)

print(f"Mean:     {amount.mean():,.4f}")
print(f"Median:   {amount.median():,.4f}")
print(f"Std dev:  {amount.std():,.4f}")
print(f"Skewness: {amount.skew():,.4f}")

,percentile,transaction_amount
0,0.0000,0.2510
1,1.0000,9.2440
2,5.0000,20.0000
3,25.0000,43.3210
4,50.0000,68.7690
5,75.0000,125.0000
6,90.0000,275.2930
7,95.0000,445.0000
8,99.0000,"1,104.0000"
9,99.9000,"2,769.8073"


Mean:     135.0272
Median:   68.7690
Std dev:  239.1625
Skewness: 14.3745


In [13]:
amount_by_target = (
    transaction
    .groupby(TARGET_COLUMN)
    .agg(
        transaction_count=(AMOUNT_COLUMN, "size"),
        total_amount=(AMOUNT_COLUMN, "sum"),
        mean_amount=(AMOUNT_COLUMN, "mean"),
        median_amount=(AMOUNT_COLUMN, "median"),
        maximum_amount=(AMOUNT_COLUMN, "max"),
    )
    .rename(index={0: "legitimate", 1: "fraud"})
)

amount_by_target["transaction_share"] = (
    amount_by_target["transaction_count"]
    / amount_by_target["transaction_count"].sum()
)

amount_by_target["amount_share"] = (
    amount_by_target["total_amount"]
    / amount_by_target["total_amount"].sum()
)

display(amount_by_target)

total_transaction_amount = transaction[AMOUNT_COLUMN].sum()
fraud_transaction_amount = transaction.loc[
    transaction[TARGET_COLUMN] == 1,
    AMOUNT_COLUMN,
].sum()

fraud_amount_share = (
    fraud_transaction_amount / total_transaction_amount
)

print(
    f"Total transaction amount:      "
    f"{total_transaction_amount:,.2f}"
)
print(
    f"Fraudulent transaction amount: "
    f"{fraud_transaction_amount:,.2f}"
)
print(
    f"Fraud share of transaction value: "
    f"{fraud_amount_share:.4%}"
)

,transaction_count,total_amount,mean_amount,median_amount,maximum_amount,transaction_share,amount_share
isFraud,,,,,,,
legitimate,569877,"76,655,103.8750",134.5117,68.5000,"31,937.3910",0.9650,0.9613
fraud,20663,"3,083,844.8600",149.2448,75.0000,"5,191.0000",0.0350,0.0387


Total transaction amount:      79,738,948.73
Fraudulent transaction amount: 3,083,844.86
Fraud share of transaction value: 3.8674%


### Business-readable amount bands

Fixed bands make the results easier to interpret operationally than percentile
groups alone.

These boundaries are exploratory and may later be replaced by payment-product
or currency-specific thresholds.


In [14]:
amount_analysis = transaction[
    [AMOUNT_COLUMN, TARGET_COLUMN]
].copy()

amount_analysis["fraud_amount"] = (
    amount_analysis[AMOUNT_COLUMN]
    * amount_analysis[TARGET_COLUMN]
)

business_band_edges = [
    0,
    25,
    50,
    100,
    200,
    500,
    1_000,
    float("inf"),
]

business_band_labels = [
    "0–<25",
    "25–<50",
    "50–<100",
    "100–<200",
    "200–<500",
    "500–<1,000",
    "1,000+",
]

amount_analysis["business_amount_band"] = pd.cut(
    amount_analysis[AMOUNT_COLUMN],
    bins=business_band_edges,
    labels=business_band_labels,
    right=False,
    include_lowest=True,
)

business_amount_summary = (
    amount_analysis
    .groupby(
        "business_amount_band",
        observed=False,
    )
    .agg(
        transaction_count=(TARGET_COLUMN, "size"),
        fraud_count=(TARGET_COLUMN, "sum"),
        total_amount=(AMOUNT_COLUMN, "sum"),
        fraud_amount=("fraud_amount", "sum"),
    )
    .reset_index()
)

business_amount_summary["fraud_rate"] = (
    business_amount_summary["fraud_count"]
    / business_amount_summary["transaction_count"]
)

business_amount_summary["transaction_share"] = (
    business_amount_summary["transaction_count"]
    / business_amount_summary["transaction_count"].sum()
)

business_amount_summary["fraud_capture_share"] = (
    business_amount_summary["fraud_count"]
    / business_amount_summary["fraud_count"].sum()
)

business_amount_summary["fraud_value_share"] = (
    business_amount_summary["fraud_amount"]
    / business_amount_summary["fraud_amount"].sum()
)

display(business_amount_summary)

,business_amount_band,transaction_count,fraud_count,total_amount,fraud_amount,fraud_rate,transaction_share,fraud_capture_share,fraud_value_share
0,0–<25,43329,3019,"709,032.2800","46,756.0570",0.0697,0.0734,0.1461,0.0152
1,25–<50,144186,4461,"5,306,136.4680","163,935.9920",0.0309,0.2442,0.2159,0.0532
2,50–<100,160742,4618,"10,608,248.2530","320,696.3210",0.0287,0.2722,0.2235,0.1040
3,100–<200,141813,3999,"17,685,725.7630","526,380.7060",0.0282,0.2401,0.1935,0.1707
4,200–<500,76146,3417,"21,565,359.3380","1,017,352.8130",0.0449,0.1289,0.1654,0.3299
5,"500–<1,000",16744,962,"11,046,874.0810","699,792.6110",0.0575,0.0284,0.0466,0.2269
6,"1,000+",7580,187,"12,817,572.5520","308,930.3600",0.0247,0.0128,0.0090,0.1002


### Amount quantiles

Quantile bands contain approximately equal transaction volumes. They help
distinguish a genuine amount-risk relationship from effects caused by heavily
unequal fixed-band sizes.


In [15]:
amount_analysis["amount_quantile"] = pd.qcut(
    amount_analysis[AMOUNT_COLUMN],
    q=10,
    duplicates="drop",
)

quantile_amount_summary = (
    amount_analysis
    .groupby(
        "amount_quantile",
        observed=False,
    )
    .agg(
        transaction_count=(TARGET_COLUMN, "size"),
        fraud_count=(TARGET_COLUMN, "sum"),
        minimum_amount=(AMOUNT_COLUMN, "min"),
        maximum_amount=(AMOUNT_COLUMN, "max"),
        total_amount=(AMOUNT_COLUMN, "sum"),
        fraud_amount=("fraud_amount", "sum"),
    )
    .reset_index()
)

quantile_amount_summary["fraud_rate"] = (
    quantile_amount_summary["fraud_count"]
    / quantile_amount_summary["transaction_count"]
)

quantile_amount_summary["fraud_capture_share"] = (
    quantile_amount_summary["fraud_count"]
    / quantile_amount_summary["fraud_count"].sum()
)

display(quantile_amount_summary)

,amount_quantile,transaction_count,fraud_count,minimum_amount,maximum_amount,total_amount,fraud_amount,fraud_rate,fraud_capture_share
0,"(0.25, 25.95]",59511,3326,0.2510,25.9500,"1,121,194.6650","54,527.4120",0.0559,0.1610
1,"(25.95, 35.95]",61650,1976,25.9600,35.9500,"1,948,118.5390","61,014.3910",0.0321,0.0956
2,"(35.95, 49.0]",65116,2100,35.9600,49.0000,"2,884,381.4660","91,282.9340",0.0323,0.1016
3,"(49.0, 57.95]",59647,1159,49.0010,57.9500,"3,244,721.4160","61,293.4340",0.0194,0.0561
4,"(57.95, 68.769]",49346,1407,57.9510,68.7580,"3,000,249.1140","86,235.3120",0.0285,0.0681
5,"(68.769, 100.0]",73349,2653,68.7800,100.0000,"6,460,951.8010","229,334.8870",0.0362,0.1284
6,"(100.0, 117.0]",72079,1423,100.0100,117.0000,"8,048,617.0270","160,129.9080",0.0197,0.0689
7,"(117.0, 159.95]",32399,1394,117.0080,159.9500,"4,667,861.8860","200,080.7990",0.0430,0.0675
8,"(159.95, 275.293]",58390,2221,159.9600,275.2930,"12,156,103.2790","463,152.6270",0.0380,0.1075
9,"(275.293, 31937.391]",59053,3004,275.4900,"31,937.3910","36,206,749.5420","1,676,793.1560",0.0509,0.1454


In [16]:
highest_business_band = (
    business_amount_summary
    .sort_values(
        ["fraud_rate", "transaction_count"],
        ascending=[False, False],
    )
    .iloc[0]
)

highest_quantile = (
    quantile_amount_summary
    .sort_values(
        ["fraud_rate", "transaction_count"],
        ascending=[False, False],
    )
    .iloc[0]
)

print(
    "Highest fixed-band fraud rate: "
    f"{highest_business_band['business_amount_band']} "
    f"({highest_business_band['fraud_rate']:.4%}, "
    f"{highest_business_band['transaction_count']:,.0f} transactions)"
)

print(
    "Highest quantile fraud rate: "
    f"{highest_quantile['amount_quantile']} "
    f"({highest_quantile['fraud_rate']:.4%}, "
    f"{highest_quantile['transaction_count']:,.0f} transactions)"
)

Highest fixed-band fraud rate: 0–<25 (6.9676%, 43,329 transactions)
Highest quantile fraud rate: (0.25, 25.95] (5.5889%, 59,511 transactions)


### Initial modeling implications

- the raw transaction amount should be preserved
- a `log1p(TransactionAmt)` feature may help with strong right skew
- amount bands may support monitoring and dashboard segmentation
- transaction count and transaction-value exposure should both be evaluated
- amount alone should not be interpreted as a fraud decision rule
- threshold selection should consider fraud value as well as fraud count
